# 7 · Property extension  (growing vocabulary of minted relations)

The relation-side twin of `4_type_extension.ipynb`. Turns one-off minted relations into a
**persistent, growing vocabulary of properties** and canonicalizes each free-form `relation_phrase`
(from nb6) against **schema.org properties ∪ property-vocabulary**. Recipe, exactly as nb4 does for
types: *define → embed → retrieve nearest existing properties → reuse the best match, else mint.*

- **Hook A** — retrieval offers vocabulary properties as candidates alongside schema.org properties,
  so the selection LLM reuses an existing property whenever one fits (else it mints a new `ex:` one).
- **schema.org-first** — when a schema.org property and a vocabulary property both fit, prefer the
  schema.org term.
- **Minting** — a minted property is admitted as-is (label + definition + `subPropertyOf` parent +
  domain/range hints). Collapsing synonymous / broader-narrower mints is deferred to the **optional
  global re-consolidation** pass at the bottom (`consolidate_vocab`).

**Reads** nb5's property pool (`schema_properties_enriched.json`) and nb6's `oie_relations.json`.
**Writes** `results/relation_vocabulary.json` and per-article `relations_ext.json` +
`relation_matching_review_ext.json`. `EXPAND_MODE` toggles grow vs. closed-schema.

In [23]:
# --- Config: property-extension stage ----------------------------------------------------
import json, re, os, copy
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

SCHEMA_DIR  = Path("schema_org_files")     # property pool from nb5
RESULTS_DIR = Path("results")              # reads oie_relations.json (nb6)
VOCAB_PATH  = RESULTS_DIR / "relation_vocabulary.json"

MATCH_MODEL  = "intfloat/e5-base-v2"       # ONE embedding model for the whole vocabulary (don't mix spaces)
GEMINI_MODEL = "gemini-3.6-flash"
EXPAND_MODE  = True                        # True: admit minted properties (grow). False: restrict to schema.org u vocab.

# No mint-time canonicalization: a minted property is admitted as-is during the incremental build.
# Collapsing synonymous / broader-narrower mints is deferred to the OPTIONAL global re-consolidation
# pass at the bottom (consolidate_vocab) — run it to clean up, or skip it to keep the raw vocabulary.

TOP_K_MATCH  = 5                           # candidate properties shown to the selection LLM (Hook A)
ALIGN_QUERY_STYLE = True                   # append "Also called <synonyms>." to each relation-phrase query

# Which articles nb7 runs on — and the order the vocabulary grows along (order matters: incremental).
#   None  -> every article that has an oie_relations.json (from nb6), sorted.
#   list  -> exactly these ids, in this order, e.g. ["art2", "art1", "art20"].
#SELECTED_ARTICLES = None
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

_available = sorted(p.parent.name for p in RESULTS_DIR.glob("*/oie_relations.json"))
if SELECTED_ARTICLES is None:
    ARTICLE_ORDER = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no oie_relations.json):", missing)
    ARTICLE_ORDER = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_ORDER, "No runnable articles selected (need nb6 oie_relations.json). Check SELECTED_ARTICLES."
print("expand:", EXPAND_MODE, "| match:", MATCH_MODEL, "| articles:", ARTICLE_ORDER)

expand: True | match: intfloat/e5-base-v2 | articles: ['art1', 'art2', 'art4', 'art5']


In [11]:
# First run only (sentence-transformers pulls torch; google-genai for the LLM stages).
%pip install -q sentence-transformers google-genai numpy

Note: you may need to restart the kernel to use updated packages.


In [12]:
# Embedding model (loaded once). e5 needs "query: " on the query side and "passage: " on documents;
# both the relation-phrase queries and every document (schema.org OR vocab property) go through the
# same convention — identical to nb4.
from sentence_transformers import SentenceTransformer
_MODEL = SentenceTransformer(MATCH_MODEL)

def _prefixes(name):
    n = name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""
QP, DP = _prefixes(MATCH_MODEL)

def embed(texts, prefix):
    return np.asarray(_MODEL.encode([prefix + t for t in texts],
                                    normalize_embeddings=True, batch_size=64))
print("embedder ready:", MATCH_MODEL, "| query_prefix", repr(QP), "| doc_prefix", repr(DP))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embedder ready: intfloat/e5-base-v2 | query_prefix 'query: ' | doc_prefix 'passage: '


In [13]:
# Schema.org PROPERTY document pool (fixed) — embedded once (from nb5). Vocabulary properties are
# embedded per article inside retrieval, since the vocabulary grows during the run.
schema_pool   = json.loads((SCHEMA_DIR / "schema_properties_enriched.json").read_text())
SCHEMA_LABELS = {p["label"] for p in schema_pool}
schema_emb    = embed([p["embedding_text"] for p in schema_pool], DP)
print(len(schema_pool), "schema.org properties embedded")

1521 schema.org properties embedded


In [14]:
# The growing property vocabulary store. One entry per canonical minted property:
#   id, canonical_label (camelCase property name), definition, parent (subPropertyOf: a schema.org OR
#   vocab property label, or null), domain[] / range[] (subject/object type hints), aliases[] (its
#   synonyms), provenance[], support, first_seen, and path (derived: super-property chain, stamped on save).
def load_vocab():
    if VOCAB_PATH.exists():
        return json.loads(VOCAB_PATH.read_text())
    return {"properties": [], "next_id": 1}

def save_vocab(vocab):
    _annotate_vocab_paths(vocab)                 # refresh each entry's derived `path` from its parent chain
    VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

def _sig(domain, range_):
    """domain → range for HUMAN display / LLM prompts (the '→' glyph is fine there)."""
    d = "/".join(domain) if domain else "ANY"
    r = "/".join(range_) if range_ else "ANY"
    return f"{d} → {r}"

def _sig_phrase(domain, range_):
    """Tokenizer-friendly signature for EMBEDDED text — spells direction in words, since e5/bert has
    no '→' token ([UNK]). Must match nb5's sig_phrase() so vocab and schema docs share one space."""
    d = " or ".join(domain) if domain else "any type"
    r = " or ".join(range_) if range_ else "any type"
    return f"from {d} to {r}"

def vocab_embedding_text(t):
    also = f" Also called {', '.join(t['aliases'])}." if t.get("aliases") else ""
    return f'{t["canonical_label"]} ({_sig_phrase(t.get("domain"), t.get("range"))}): {t.get("definition", "")}{also}'

def _find_vocab_by_label(vocab, label):
    for t in vocab["properties"]:
        if t["canonical_label"] == label or label in t.get("aliases", []):
            return t
    return None

def _is_vocab_label(vocab, label):
    return _find_vocab_by_label(vocab, label) is not None

def _bump(ent, aid, phrase, instances):
    ent["support"] = ent.get("support", 0) + len(instances)
    ent.setdefault("provenance", []).append(
        {"article": aid, "relation_phrase": phrase,
         "instances": [f'{i["subject_text"]} -> {i["object_text"]}' for i in instances]})

In [15]:
# Hook A — retrieve top-k candidate properties for each relation phrase against schema.org u current
# vocabulary. Both sources expose their domain->range `signature` and their `subPropertyOf` parent, so
# the selection LLM can judge direction/domain-range fit and reach for either as a mint parent.
_SCHEMA_BY_LABEL = {p["label"]: p for p in schema_pool}

def _schema_super(label):
    """Immediate super-property (subPropertyOf) of a schema.org property, or None. schema.org lists
    sub_property_of as a (usually empty) list; we take the first schema.org super-property."""
    p = _SCHEMA_BY_LABEL.get(label)
    for sup in (p.get("sub_property_of") if p else []) or []:
        if sup in SCHEMA_LABELS:
            return sup
    return None

def _schema_super_chain(label):
    """Top-down super-property chain INCLUDING `label`: [root_super, ..., label]. Sparse for
    properties (most schema.org properties have no super-property), so usually just [label]."""
    chain, cur, seen = [], label, set()
    while cur and cur not in seen:
        seen.add(cur); chain.append(cur); cur = _schema_super(cur)
    return list(reversed(chain))

def vocab_ancestor_path(t, vocab):
    """super-property chain for a vocab property (top-down, ending at the immediate parent, EXCLUDING
    the property itself). Climbs `parent` (subPropertyOf) links up through the vocabulary, then splices
    in the schema.org super-chain of the first schema property it reaches. A `seen` guard prevents loops."""
    up, cur, seen = [], t.get("parent"), set()
    while cur and cur not in seen:
        seen.add(cur)
        ent = _find_vocab_by_label(vocab, cur)
        if ent:                                   # another vocab property -> keep climbing the vocab tree
            up.append(cur); cur = ent.get("parent")
        else:                                     # a schema.org property -> splice its super-chain in
            up.extend(reversed(_schema_super_chain(cur))); cur = None
    return " > ".join(reversed(up))

def _annotate_vocab_paths(vocab):
    """Stamp each entry's derived `path` (super-property chain, ending at the immediate parent).
    Derived from the `parent` chains (the source of truth) — refreshed on every save."""
    for t in vocab["properties"]:
        t["path"] = vocab_ancestor_path(t, vocab)
    return vocab

def _full_path(tref, vocab):
    """Full super-property path for a resolved property reference (used in the review's chosen block)."""
    if tref.get("source") == "vocab" and tref.get("vocab_id"):
        ent = next((t for t in vocab["properties"] if t["id"] == tref["vocab_id"]), None)
        if ent is not None:
            return vocab_ancestor_path(ent, vocab)
    if tref.get("source") == "schema.org" and tref.get("label"):
        return " > ".join(_schema_super_chain(tref["label"])[:-1])   # super-properties only
    return None

def retrieve_candidates(phrase_texts, vocab):
    q_emb = embed(phrase_texts, QP)
    meta  = [("schema.org", p) for p in schema_pool]
    d_emb = schema_emb
    if vocab["properties"]:
        v_emb = embed([vocab_embedding_text(t) for t in vocab["properties"]], DP)
        d_emb = np.vstack([schema_emb, v_emb])
        meta  = meta + [("vocab", t) for t in vocab["properties"]]
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :TOP_K_MATCH]
    out = []
    for ti in range(len(phrase_texts)):
        row = []
        for j in topk[ti]:
            src, obj = meta[j]
            if src == "schema.org":
                row.append({"source": "schema.org", "label": obj["label"], "signature": obj.get("signature", ""),
                            "parent": _schema_super(obj["label"]), "score": round(float(sims[ti, j]), 3),
                            "comment": obj.get("enriched_comment") or obj.get("comment", "")})
            else:
                row.append({"source": "vocab", "id": obj["id"], "label": obj["canonical_label"],
                            "signature": _sig(obj.get("domain"), obj.get("range")), "parent": obj.get("parent"),
                            "score": round(float(sims[ti, j]), 3), "comment": obj.get("definition", "")})
        out.append(row)
    return out

In [16]:
# Article inputs + Gemini client (JSON mode, temperature 0 for reproducibility).
def load_article(aid):
    """Read nb6's oie_relations.json and group relation instances by their free-form relation_phrase.
    Returns (oie, by_id entity map, phrases[], query_texts[], groups{phrase: [instances]})."""
    oie   = json.loads((RESULTS_DIR / aid / "oie_relations.json").read_text())
    by_id = {e["id"]: e for e in oie.get("entities", [])}
    rinfo = {t["relation_phrase"]: t for t in oie.get("relation_types", [])}
    phrases = list(dict.fromkeys(r["relation_phrase"] for r in oie["relations"]))
    def qtext(ph):
        info = rinfo.get(ph, {})
        q = f'{ph}: {info.get("definition", "")}'.strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f'{q} Also called {", ".join(info["synonyms"])}.'
        return q
    groups = {}
    for r in oie["relations"]:
        groups.setdefault(r["relation_phrase"], []).append(r)
    return oie, by_id, rinfo, phrases, [qtext(p) for p in phrases], groups

def _etype(by_id, eid):
    return (by_id.get(eid) or {}).get("type")

def group_signatures(instances, by_id):
    """Distinct observed subject_type -> object_type signatures within a relation-phrase group."""
    sigs = Counter(f'{_etype(by_id, i["subject_id"])} → {_etype(by_id, i["object_id"])}' for i in instances)
    return [s for s, _ in sigs.most_common()]

from google import genai
from google.genai import types as genai_types
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_json(system, user):
    r = client.models.generate_content(
        model=GEMINI_MODEL, contents=user,
        config=genai_types.GenerateContentConfig(
            system_instruction=system, temperature=0, response_mime_type="application/json"))
    return r.text

def parse_json(text):
    s = (text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s); s = re.sub(r"\s*```$", "", s)
    return json.loads(s)

## Inspect retrieval — top-k candidates by embedding similarity (no LLM)

Preview **Hook A on its own**: for one article, show the top-`TOP_K_MATCH` candidate properties each
relation phrase retrieves from **schema.org ∪ current property-vocabulary**, ranked by cosine
similarity — *before* the LLM judges or mints anything. On a fresh run the vocabulary is empty, so the
candidates are schema.org-only; re-run this cell after **Build** (below) to see minted `ex:` properties
surface as candidates too. No Gemini call — pure retrieval.

In [17]:
# Retrieval-only preview: the ranked candidates per relation phrase, with NO Gemini call. Uses the
# CURRENT vocab state (load_vocab(); empty => schema.org-only). This is exactly the `cands` that would
# be handed to the LLM judge in process_article — inspect it first to sanity-check what the embedder
# surfaces. Set INSPECT_ARTICLE to any id in ARTICLE_ORDER.
INSPECT_ARTICLE = ARTICLE_ORDER[0]

def preview_retrieval(aid, vocab=None, top_k=TOP_K_MATCH, show_comment=True):
    vocab = load_vocab() if vocab is None else vocab
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands = retrieve_candidates(qtexts, vocab)
    print(f"{aid}: {len(phrases)} relation phrases | vocab has {len(vocab['properties'])} minted "
          f"property(ies)\n" + "=" * 92)
    for i, ph in enumerate(phrases):
        sigs = ", ".join(group_signatures(groups[ph], by_id))
        print(f'\n"{ph}"   ({len(groups[ph])} instance(s) | observed {sigs})')
        print(f'   query: {qtexts[i]}')
        for k, c in enumerate(cands[i][:top_k], 1):
            print(f'   {k}. [{c["source"]:9}] {c["label"]:<26} score={c["score"]:.3f}   sig: {c.get("signature","")}')
            if show_comment and c.get("comment"):
                print(f'          {c["comment"][:100]}')
    return cands

_ = preview_retrieval(INSPECT_ARTICLE)

art1: 8 relation phrases | vocab has 0 minted property(ies)

"recorded"   (3 instance(s) | observed Country → QuantitativeValue)
   query: recorded: Officially registered or documented a measured figure, occurrence, or statistic. Also called registered, reported, documented, logged.
   1. [schema.org] variableMeasured           score=0.801   sig: Dataset/Observation → Property/PropertyValue/StatisticalVariable/Text
          Relates a dataset or observation to the specific variable, property, or statistical concept that is 
   2. [schema.org] legislationPassedBy        score=0.798   sig: Legislation → Organization/Person
          This property identifies the organization or person that legally enacted or passed a subject legisla
   3. [schema.org] measuredProperty           score=0.797   sig: Observation/StatisticalVariable → Property
          This property identifies the specific characteristic or property that is being measured or observed 
   4. [schema.org] recordedIn            

In [18]:
# ==== EDIT ME — prompt ===================================================================
# PROPERTY_SELECT_PROMPT is the property-side twin of nb4's TYPE selection task: canonicalize each
# free-form relation phrase to a schema.org OR vocabulary property, else mint a new ex: property.
# (Relation-instance dedup is deterministic — collapsing identical subject/object pairs onto one
# triple — so it is done in code, not by the LLM; this prompt only chooses the property.)
PROPERTY_SELECT_PROMPT = """You are an expert knowledge-engineering assistant. You are given several groups of RELATION instances from one document. Each group shares a single free-form relation phrase (an OIE relation label). For each group independently, choose the single best PROPERTY to canonicalize that relation to — do NOT merge or compare instances across different groups.

Each group provides:
- RELATION_PHRASE: the free-form relation label these instances were given, plus its definition. A relation reads subject --RELATION_PHRASE--> object.
- OBSERVED_SIGNATURES: the subject-type -> object-type signatures actually seen among the instances (helps you check domain/range fit and direction).
- CANDIDATES: a ranked list of candidate properties proposed as matches. Each candidate is either an existing Schema.org property or an existing property from our extended vocabulary; its source is marked [schema.org] or [vocab]. Each shows its `signature` (domain -> range, i.e. the subject types it applies to and the object types it accepts) and its `parent` (its super-property via subPropertyOf, if any).
- INSTANCES: the actual subject --> object pairs (with their entity types) that were assigned this relation phrase.

Choose the single best property for the group.
- Prefer an existing candidate. Pick the one whose meaning (from its comment) and DIRECTION and domain->range signature best fit the relation phrase and the instances. When a Schema.org candidate and a vocabulary candidate both fit equally well, PREFER THE SCHEMA.ORG ONE (schema.org-first).
- Direction matters: the candidate must express the relation in the SAME direction as subject --> object. If a candidate expresses the inverse relation (object --> subject), it does NOT fit — reject it (unless you also flip; but do not flip: keep the phrase's direction and reject inverse-only candidates).
- Instance test — apply before accepting any candidate, judging by the INSTANCES, not the phrase definition alone: would each subject/object pair individually be a valid instance of that property, with the subject fitting its domain and the object fitting its range, and sitting naturally alongside the examples in the candidate's comment? If the pairs do not fit the candidate's domain/range or direction, the candidate is wrong; reject it. A candidate that is somewhat broader but still preserves what makes the relation distinctive is preferable to a new property.
- Mint a new property when no candidate genuinely fits (including when the best candidate is only "correct" by being so broad it loses the relation's distinguishing meaning, or points the wrong direction). Mint a concise camelCase property label (e.g. "operatesFlightTo", "electedTo") and a one-sentence definition stating what the subject is to the object.
- When you mint, also assign: a parent (the closest existing property this should be a subPropertyOf — a Schema.org OR [vocab] candidate label, or any candidate's `parent`; use null if none is a sensible super-property), and domain / range (short lists of the subject types and object types the minted property applies to, taken from the OBSERVED_SIGNATURES).
- Report: source ("schema.org" or "vocab" when you chose a candidate, matching that candidate's marked source; "new" when you mint), the chosen label (the exact candidate label, or the new camelCase label), whether it is new, its definition / parent / domain / range (only when new; else null), and a one-line justification.

Return ONLY valid JSON — no prose, no code fences. Output one result object per input group, in the same order, each echoing its exact relation_phrase string. Angle-bracket placeholders show the shape, not literal values:
{"results": [{"relation_phrase": "<phrase>", "chosen_property": {"source": "schema.org | vocab | new", "label": "<property label>", "is_new": false, "definition": null, "parent": null, "domain": null, "range": null, "justification": "one short line"}}]}
Include every group exactly once."""
print("prompt loaded")

prompt loaded


In [19]:
# Hook A caller — ONE Gemini call per article: choose a property for every relation-phrase group.
def llm_select(phrases, rinfo, groups, by_id, cands):
    blocks = []
    for i, ph in enumerate(phrases):
        b = [f"GROUP {i+1} —", f'RELATION_PHRASE: "{ph}"']
        if rinfo.get(ph, {}).get("definition"):
            b.append(f'DEFINITION: {rinfo[ph]["definition"]}')
        b.append("OBSERVED_SIGNATURES: " + "; ".join(group_signatures(groups[ph], by_id)))
        b.append("CANDIDATES (ranked):")
        if cands[i]:
            for k, c in enumerate(cands[i], 1):
                par = c.get("parent") or "—"
                b.append(f'  {k}. [{c["source"]}] {c["label"]} | signature: {c.get("signature","")} | subPropertyOf: {par}')
                b.append(f'     {c.get("comment", "")}')
        else:
            b.append("  (none)")
        b.append("INSTANCES:")
        b += [f'  - "{r["subject_text"]}" ({_etype(by_id, r["subject_id"])}) --> "{r["object_text"]}" ({_etype(by_id, r["object_id"])})'
              for r in groups[ph]]
        blocks.append("\n".join(b))
    user = f"You are given {len(phrases)} GROUPS from one document.\n\n" + "\n\n".join(blocks)
    parsed  = parse_json(gemini_json(PROPERTY_SELECT_PROMPT, user))
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    out = {r.get("relation_phrase"): (r.get("chosen_property") or {})
           for r in (results or []) if r.get("relation_phrase") in set(phrases)}
    for i, ph in enumerate(phrases):                 # fallback: omitted group -> top candidate
        if ph not in out:
            top = cands[i][0] if cands[i] else {"source": "schema.org", "label": "relatedTo"}
            out[ph] = {"source": top["source"], "label": top["label"], "is_new": False,
                       "justification": "fallback: top candidate"}
    return out

# ---- Minting: admit a chosen "new" property into the vocabulary -------------------------------
def _schema_ref(label):
    return {"label": label, "source": "schema.org", "vocab_id": None,
            "definition": None, "parent": None, "domain": None, "range": None, "is_new_to_vocab": False}

def _vocab_ref(ent, is_new=False):
    return {"label": ent["canonical_label"], "source": "vocab", "vocab_id": ent["id"],
            "definition": ent["definition"], "parent": ent["parent"],
            "domain": ent.get("domain"), "range": ent.get("range"), "is_new_to_vocab": is_new}

# Admit a minted property as a NEW vocabulary entry (or, in restrict mode, fall back to its parent /
# a generic related-to link). Parent (subPropertyOf) may be null — properties have no single root.
def _apply_mint(aid, phrase, ch, group, vocab):
    label = ch.get("label"); definition = ch.get("definition") or ""
    parent = ch.get("parent")
    if parent is not None and not (parent in SCHEMA_LABELS or _is_vocab_label(vocab, parent)):
        parent = None                                          # unknown parent -> no super-property
    if not EXPAND_MODE:                                        # restrict mode: fall back to the parent
        return _schema_ref(parent) if parent in SCHEMA_LABELS else _schema_ref("relatedTo")
    ent = {"id": f"P{vocab['next_id']:04d}", "canonical_label": label, "definition": definition,
           "parent": parent, "domain": ch.get("domain") or [], "range": ch.get("range") or [],
           "aliases": [], "support": len(group), "first_seen": aid,
           "provenance": [{"article": aid, "relation_phrase": phrase,
                           "instances": [f'{i["subject_text"]} -> {i["object_text"]}' for i in group]}]}
    vocab["properties"].append(ent); vocab["next_id"] += 1
    return _vocab_ref(ent, is_new=True)

In [20]:
# Build canonical relation records. Every nb6 relation instance is mapped to its group's chosen
# property, then instances are DEDUPED deterministically on (subject_id, chosen_property, object_id):
# identical triples (e.g. two phrases that both canonicalize to the same property between the same two
# entities) collapse into one record whose evidence is the union of the spans. No LLM here.
def _build_relation_records(oie, by_id, phrases, groups, resolved):
    records = {}   # (subject_id, property_label, object_id) -> record
    for ph in phrases:
        tref = resolved[ph]
        for r in groups[ph]:
            key = (r["subject_id"], tref.get("label"), r["object_id"])
            rec = records.get(key)
            if rec is None:
                rec = {
                    "subject_id":   r["subject_id"],
                    "subject_text": r["subject_text"],
                    "subject_type": _etype(by_id, r["subject_id"]),
                    "property":                 tref.get("label"),
                    "property_source":          tref.get("source"),
                    "vocab_id":                 tref.get("vocab_id"),
                    "property_is_new_to_vocab": tref.get("is_new_to_vocab", False),
                    "property_definition":      tref.get("definition"),
                    "property_parent":          tref.get("parent"),
                    "property_justification":   tref.get("justification"),
                    "object_id":    r["object_id"],
                    "object_text":  r["object_text"],
                    "object_type":  _etype(by_id, r["object_id"]),
                    "relation_phrases": [],
                    "evidence":         [],
                    "confidence":       0.0,
                }
                records[key] = rec
            if ph not in rec["relation_phrases"]:
                rec["relation_phrases"].append(ph)
            ev = r.get("evidence")
            if ev and ev not in rec["evidence"]:
                rec["evidence"].append(ev)
            rec["confidence"] = max(rec["confidence"], float(r.get("confidence") or 0.0))
    return list(records.values())

In [21]:
# Process ONE article end-to-end, self-contained:
#   retrieve (Hook A) -> ONE LLM call to choose a property per relation-phrase group -> admit any
#   minted properties as new vocabulary entries -> build canonical (deduped) relation records.
#   Writes relations_ext.json AND relation_matching_review_ext.json (per relation phrase: query text,
#   instances, the candidates the LLM chose from, and the final chosen property + its super-property path).
def process_article(aid, vocab, write=True, save=True, verbose=True):
    oie, by_id, rinfo, phrases, qtexts, groups = load_article(aid)
    cands  = retrieve_candidates(qtexts, vocab)
    report = llm_select(phrases, rinfo, groups, by_id, cands)

    resolved, n_mint = {}, 0
    for ph in phrases:
        ch = report[ph]
        is_mint = ch.get("is_new") or ch.get("source") == "new"
        if is_mint:                                                # admit the mint as a new vocab entry
            before = len(vocab["properties"])
            tref = _apply_mint(aid, ph, ch, groups[ph], vocab)
            n_mint += (len(vocab["properties"]) > before)
        else:
            ent = _find_vocab_by_label(vocab, ch.get("label"))     # chose an existing property
            if ent:
                _bump(ent, aid, ph, groups[ph]); tref = _vocab_ref(ent)
            else:
                tref = _schema_ref(ch.get("label"))
        tref["justification"] = ch.get("justification")
        resolved[ph] = tref

    records = _build_relation_records(oie, by_id, phrases, groups, resolved)

    if write:
        out_dir = RESULTS_DIR / aid
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "relations_ext.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE,
             "entities": oie.get("entities", []), "relations": records},
            indent=2, ensure_ascii=False))
        # Review file: one row per relation phrase with the query text, instances, the candidates the
        # LLM chose from (at THIS article's vocab state), the final chosen property (with super-property
        # path), and the LLM's original proposal for auditing.
        review = [{
            "relation_phrase":      ph,
            "definition":           rinfo.get(ph, {}).get("definition"),
            "embedding_query_text": qtexts[i],
            "observed_signatures":  group_signatures(groups[ph], by_id),
            "chosen_property": {
                "label":            resolved[ph].get("label"),
                "source":           resolved[ph].get("source"),
                "vocab_id":         resolved[ph].get("vocab_id"),
                "is_new_to_vocab":  resolved[ph].get("is_new_to_vocab", False),
                "definition":       resolved[ph].get("definition"),
                "parent":           resolved[ph].get("parent"),
                "domain":           resolved[ph].get("domain"),
                "range":            resolved[ph].get("range"),
                "path":             _full_path(resolved[ph], vocab),
                "justification":    resolved[ph].get("justification"),
            },
            "llm_proposed":         {k: (report[ph] or {}).get(k) for k in ("source", "label", "is_new")},
            "instances":            [f'{r["subject_text"]} -> {r["object_text"]}' for r in groups[ph]],
            "candidates":           cands[i],
        } for i, ph in enumerate(phrases)]
        (out_dir / "relation_matching_review_ext.json").write_text(json.dumps(
            {"article_id": aid, "match_model": MATCH_MODEL, "expand_mode": EXPAND_MODE, "relations": review},
            indent=2, ensure_ascii=False))
    if save:
        save_vocab(vocab)
    if verbose:
        merged = sum(1 for r in records if len(r["relation_phrases"]) > 1 or len(r["evidence"]) > 1)
        n_inst = len(oie["relations"])
        print(f"  {aid}: {len(phrases)} phrases, {n_inst} instances -> {len(records)} triples "
              f"({merged} merged) | +{n_mint} minted | vocab now {len(vocab['properties'])}")
    return records, vocab

## Dry run — one article, no writes

In [ ]:
# # Dry run on the FIRST article against a COPY of the current vocab (no writes, no vocab mutation).
# _vocab_preview = copy.deepcopy(load_vocab())
# _records, _ = process_article(ARTICLE_ORDER[0], _vocab_preview, write=False, save=False)
# for (src, lbl), c in Counter((r.get("property_source"), r["property"]) for r in _records).most_common():
#     print(f"  {str(src):<11} {lbl}  x{c}")

## Build — incremental vocabulary over all articles (in order)

In [24]:
# Incremental build. FRESH=True: start from an EMPTY vocab and process ALL of ARTICLE_ORDER (reproducible).
# FRESH=False: keep the existing relation_vocabulary.json and process only NEW_ARTICLES (future growth).
FRESH        = True
NEW_ARTICLES = []          # used only when FRESH=False

if FRESH:
    vocab = {"properties": [], "next_id": 1}; save_vocab(vocab); todo = ARTICLE_ORDER
else:
    vocab = load_vocab()
    todo  = NEW_ARTICLES or [a for a in ARTICLE_ORDER
                             if not (RESULTS_DIR / a / "relations_ext.json").exists()]

print(f"building property vocabulary over {len(todo)} article(s): {todo}\n")
for aid in todo:
    process_article(aid, vocab, write=True, save=True, verbose=True)
print(f"\ndone — {len(vocab['properties'])} canonical properties in {VOCAB_PATH}")

building property vocabulary over 4 article(s): ['art1', 'art2', 'art4', 'art5']

  art1: 8 phrases, 12 instances -> 12 triples (0 merged) | +6 minted | vocab now 6
  art2: 11 phrases, 14 instances -> 14 triples (0 merged) | +5 minted | vocab now 11
  art4: 5 phrases, 20 instances -> 20 triples (0 merged) | +3 minted | vocab now 14
  art5: 51 phrases, 93 instances -> 93 triples (0 merged) | +45 minted | vocab now 59

done — 59 canonical properties in results/relation_vocabulary.json


## Report + verify

In [ ]:
# Property vocabulary summary, then verify each ext file covers every nb6 relation instance
# (each oie relation maps to exactly one canonical triple — none lost; merges only collapse duplicates).
vocab = load_vocab()
print(f"PROPERTY VOCABULARY — {len(vocab['properties'])} canonical properties\n" + "-" * 74)
for t in sorted(vocab["properties"], key=lambda x: -x.get("support", 0)):
    al = f"  aka {t['aliases']}" if t.get("aliases") else ""
    print(f"  {t['id']}  {t['canonical_label']:<26} subPropertyOf {str(t.get('parent')):<16} "
          f"support={t.get('support')} | {_sig(t.get('domain'), t.get('range'))}{al}")

print("\nVERIFY ext vs oie_relations (every relation instance represented)\n" + "-" * 74)
ok = True
for aid in ARTICLE_ORDER:
    ext = RESULTS_DIR / aid / "relations_ext.json"
    if not ext.exists():
        print(f"  {aid:<8} (no ext file)"); continue
    oie = json.loads((RESULTS_DIR / aid / "oie_relations.json").read_text())
    x   = json.loads(ext.read_text())["relations"]
    # every (subject_id, object_id) pair from oie must appear in some record (property-agnostic check)
    orig_pairs = {(r["subject_id"], r["object_id"]) for r in oie["relations"]}
    got_pairs  = {(r["subject_id"], r["object_id"]) for r in x}
    nvoc = sum(1 for r in x if r.get("property_source") == "vocab")
    covered = orig_pairs <= got_pairs
    ok &= covered
    print(f"  {aid:<8} {len(oie['relations']):>4} instances -> {len(x):>4} triples "
          f"({nvoc} via vocab)   {'OK' if covered else 'FAIL: pair(s) dropped'}")
print("-" * 74); print("consistent" if ok else "inconsistencies")

# ─────────────────────────── GLOBAL RE-CONSOLIDATION (optional) ───────────────────────────
## Everything below is the optional global-reconsolidation block

The property-side twin of nb4's re-consolidation. Run the pipeline above (through **Build** and
**Report + verify**) to produce the raw incremental property vocabulary (`relation_vocabulary.json`).
The cells below are a **self-contained, optional block** — run them to clean the vocabulary globally
(merge synonymous properties, or link a specific property under a broader one as `subPropertyOf`), or
**skip them entirely** to use the raw vocabulary as-is. This block writes only **NEW files** — every
incremental output is left untouched, so you can always compare before/after.

Same scale caveat as nb4: this works while the whole extension vocabulary fits in one prompt. If it
grows into the many hundreds/thousands, reintroduce embeddings to shortlist candidate pairs.

The block, in order:
1. **Config + prompt** — `SUBPROP_MODE`, output paths, `load_consolidated`, `CONSOLIDATE_ALL_PROMPT`.
2. **`consolidate_vocab`** — ONE Gemini call → `relation_vocabulary_consolidated.json` + `relation_consolidation_report.json`.
3. **`refresh_ext_after_consolidation`** — no LLM; writes `relations_ext_consolidated.json` per article + a changes-only `relation_consolidation_ext_changes.json`.
4. **`build_property_hierarchy`** — indented schema.org + extension property tree, into `property_hierarchy_ext.txt` (raw) and `property_hierarchy_ext_consolidated.txt` (consolidated).

In [ ]:
# Global consolidation prompt — used by the ONE-call consolidate_vocab below.
# SUBPROP_MODE controls how a SPECIFIC property is related to a broader property ALREADY in the vocab:
#   "off"        : only merge exact synonyms (no broader/narrower handling).
#   "subproperty": KEEP the specific property, set its parent (subPropertyOf) to the broader vocab
#                  property — builds an is-a-kind-of-relation hierarchy; preserves granularity.
#   "merge"      : COLLAPSE the specific property into the broader parent (child removed, folded in).
SUBPROP_MODE = "subproperty"
assert SUBPROP_MODE in ("off", "subproperty", "merge"), "SUBPROP_MODE must be 'off', 'subproperty', or 'merge'"

# Consolidation writes to SEPARATE files — the raw relation_vocabulary.json is left INTACT.
CONSOLIDATED_VOCAB_PATH   = RESULTS_DIR / "relation_vocabulary_consolidated.json"   # consolidated snapshot
CONSOLIDATION_REPORT_PATH = RESULTS_DIR / "relation_consolidation_report.json"      # what merged / re-parented
def load_consolidated():
    """The consolidated vocab if it exists, else fall back to the raw incremental one."""
    return json.loads(CONSOLIDATED_VOCAB_PATH.read_text()) if CONSOLIDATED_VOCAB_PATH.exists() else load_vocab()

CONSOLIDATE_ALL_PROMPT = """You are organizing a vocabulary of RELATION properties. You are given the full list of properties (id, label, parent, support, domain, range, definition). Each property expresses a directed relation subject --property--> object. Produce GROUPS of two kinds:

SYNONYM — two or more entries that denote the SAME relation (true synonyms: the same relation in the same direction, differing only in wording or label). Set keep_id to any member and merge_ids to the rest.
SUBPROPERTY — one or more SPECIFIC (child) properties that are all a more specific kind of a single more GENERAL (parent) relation WHICH IS ALSO PRESENT in this vocabulary — an is-a relationship between relations, in the SAME direction. Set keep_id to the GENERAL parent property's id and merge_ids to the specific child ones. Only do this when the parent property already appears in the list (never invent one) and genuinely subsumes the children in the same direction — every instance under a child must still be a valid instance of the parent relation.

Rules:
- Judge by each property's LABEL, DEFINITION, and domain/range direction (not the label alone). Be conservative. For SYNONYM, only group true equivalents in the same direction (never group a relation with its inverse). For SUBPROPERTY, only relate a child to a parent when the parent clearly and safely subsumes it in the same direction; if a real distinction would be lost, leave them separate.
- Each property belongs to at most one group. Every group has a keep_id and one or more merge_ids. Give a one-line reason and set "kind" to "synonym" or "subproperty".

Return ONLY valid JSON — no prose, no code fences. Angle-bracket placeholders show the shape, not literal values:
{"groups": [{"kind": "subproperty", "keep_id": "<parent id>", "merge_ids": ["<child id>"], "reason": "one short line"}, {"kind": "synonym", "keep_id": "<id>", "merge_ids": ["<id>"], "reason": "one short line"}]}"""
print(f"consolidation prompt loaded | SUBPROP_MODE={SUBPROP_MODE!r} | consolidated -> {CONSOLIDATED_VOCAB_PATH.name}")

In [ ]:
import time

def consolidate_vocab(vocab, write_report=True, verbose=True):
    """Global re-consolidation of the PROPERTY vocabulary in ONE Gemini call. The model returns
    SYNONYM and SUBPROPERTY groups; how SUBPROPERTY (is-a) groups are applied depends on SUBPROP_MODE:
      "off"         -> ignore them (only synonyms merge).
      "subproperty" -> KEEP each child, set its parent (subPropertyOf) to the broader property.
      "merge"       -> fold each child into the broader parent (removed; label -> alias).
    SYNONYM groups always merge onto the highest-support member. Writes the consolidated vocab to
    CONSOLIDATED_VOCAB_PATH and a report to CONSOLIDATION_REPORT_PATH — the raw relation_vocabulary.json
    is NOT modified (pass load_vocab() as `vocab`)."""
    ents  = list(vocab["properties"])
    by_id = {t["id"]: t for t in ents}

    def _articles_of(t):
        return [p.get("article") for p in t.get("provenance", []) if p.get("article")]
    articles_before = sorted({a for t in ents for a in _articles_of(t)})

    groups = []
    if len(ents) >= 2:
        lines = [f'- id: {t["id"]} | label: {t["canonical_label"]} | parent: {t.get("parent")} '
                 f'| support: {t.get("support", 0)} | {_sig(t.get("domain"), t.get("range"))} '
                 f'| def: {t.get("definition", "")}'
                 + (f' | aka: {", ".join(t["aliases"])}' if t.get("aliases") else "")
                 for t in sorted(ents, key=lambda x: x["id"])]
        gate = ("\nBoth SYNONYM and SUBPROPERTY groups are enabled." if SUBPROP_MODE != "off" else
                "\nReturn only SYNONYM groups; do NOT relate broader/narrower (subproperty) properties.")
        try:
            res = parse_json(gemini_json(CONSOLIDATE_ALL_PROMPT + gate, "VOCABULARY:\n" + "\n".join(lines)))
            groups = (res.get("groups", res) if isinstance(res, dict) else res) or []
        except Exception as ex:
            print(f"[WARN] consolidation call failed ({type(ex).__name__}: {ex})")

    merges, reparents, merged_ids = [], [], set()
    for g in groups:
        kind = (g.get("kind") or "synonym").lower()
        keep = g.get("keep_id")
        ids  = list(dict.fromkeys(i for i in ([keep] + (g.get("merge_ids") or []))
                                  if i in by_id and i not in merged_ids))
        if len(ids) < 2:
            continue
        is_subprop = kind.startswith("subprop")

        # SUBPROPERTY + subproperty mode: keep the children, just re-parent them onto the broader property
        if is_subprop and SUBPROP_MODE == "subproperty":
            if keep not in by_id or keep in merged_ids:
                continue
            parent = by_id[keep]
            rec = {"kind": "subproperty", "action": "subproperty", "parent_id": parent["id"],
                   "parent_label": parent["canonical_label"], "reason": g.get("reason"), "children": []}
            for cid in ids:
                if cid == keep:
                    continue
                child = by_id[cid]
                if child.get("parent") == parent["canonical_label"]:
                    continue                                      # already under this parent — no-op
                rec["children"].append({"id": cid, "label": child["canonical_label"],
                                        "old_parent": child.get("parent")})
                child["parent"] = parent["canonical_label"]       # keep the child; re-parent it
            if rec["children"]:
                reparents.append(rec)
            continue

        # otherwise MERGE: synonyms always; subproperty only when SUBPROP_MODE == "merge"
        if is_subprop and SUBPROP_MODE != "merge":
            continue
        members = [by_id[i] for i in ids]
        if is_subprop:
            if keep not in by_id or keep in merged_ids:
                continue
            survivor = by_id[keep]                                # broader parent survives
        else:
            survivor = max(members, key=lambda t: (t.get("support", 0), -int(t["id"][1:])))
        rec = {"kind": kind, "action": "merge", "survivor_id": survivor["id"],
               "survivor_label": survivor["canonical_label"], "reason": g.get("reason"), "merged": [],
               "articles": sorted({a for t in members for a in _articles_of(t)})}
        for m in members:
            if m["id"] == survivor["id"]:
                continue
            for lbl in [m["canonical_label"]] + m.get("aliases", []):
                if lbl != survivor["canonical_label"] and lbl not in survivor["aliases"]:
                    survivor["aliases"].append(lbl)
            survivor["support"] = survivor.get("support", 0) + m.get("support", 0)
            survivor.setdefault("provenance", []).extend(m.get("provenance", []))
            rec["merged"].append({"id": m["id"], "label": m["canonical_label"],
                                  "support": m.get("support", 0), "articles": _articles_of(m)})
            merged_ids.add(m["id"])
        if rec["merged"]:
            rec["resulting_support"] = survivor["support"]
            rec["resulting_aliases"] = list(survivor["aliases"])
            merges.append(rec)

    if merged_ids:
        vocab["properties"] = [t for t in vocab["properties"] if t["id"] not in merged_ids]

    # write the CONSOLIDATED vocabulary to its own file (raw relation_vocabulary.json stays untouched)
    _annotate_vocab_paths(vocab)                 # refresh derived `path`s (re-parenting/merges changed them)
    CONSOLIDATED_VOCAB_PATH.write_text(json.dumps(vocab, indent=2, ensure_ascii=False))

    n_reparented = sum(len(r["children"]) for r in reparents)
    report = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "match_model": MATCH_MODEL,
        "gemini_model": GEMINI_MODEL, "expand_mode": EXPAND_MODE,
        "subprop_mode": SUBPROP_MODE, "source_vocab": str(VOCAB_PATH.name),
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "articles_used": articles_before, "n_articles": len(articles_before),
        "vocab_size_before": len(ents), "vocab_size_after": len(vocab["properties"]),
        "n_merges": len(merges), "n_props_removed": len(merged_ids), "n_reparented": n_reparented,
        "merges": merges, "reparents": reparents,
    }
    if write_report:
        CONSOLIDATION_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    if verbose:
        print(f"consolidation (1 call): subprop_mode={SUBPROP_MODE!r} | {len(merges)} merge-group(s), "
              f"{len(merged_ids)} removed, {n_reparented} re-parented | vocab {len(ents)} -> {len(vocab['properties'])}")
        for r in merges:
            print(f"  [merge/{r['kind'][:4]}] keep {r['survivor_id']} {r['survivor_label']!r} <= "
                  f"{[m['label'] for m in r['merged']]}")
        for r in reparents:
            print(f"  [subproperty] {[c['label'] for c in r['children']]} subPropertyOf {r['parent_label']}")
        print(f"  consolidated vocab -> {CONSOLIDATED_VOCAB_PATH}  (raw {VOCAB_PATH.name} untouched)")
        if write_report:
            print(f"  report -> {CONSOLIDATION_REPORT_PATH}")
    return report

# Runs on the RAW incremental vocab and writes the consolidated result to a separate file.
_ = consolidate_vocab(load_vocab())

### Refresh `relations_ext.json` after consolidation → **separate** files — no LLM

Apply the consolidation's `merged_id → survivor` mapping (and subproperty re-parentings) to the
per-article files deterministically, writing the result to a **new** `relations_ext_consolidated.json`
per article — the incremental `relations_ext.json` is left untouched. Also emit one global
`relation_consolidation_ext_changes.json` listing **only** the records that changed (before → after).

In [ ]:
# Produce a CONSOLIDATED copy of each per-article relations_ext.json WITHOUT the LLM — the incremental
# relations_ext.json files are left UNTOUCHED. Two effects, applied only to the copy:
#   - MERGED properties: records pointing at a removed vocab_id are remapped to its survivor.
#   - RE-PARENTED properties (subproperty mode): the property survives, so we re-sync every vocab-typed
#     record's label / definition / parent from the CONSOLIDATED vocabulary (picks up the new parent).
# `property_is_new_to_vocab` is a BUILD-TIME provenance fact — consolidation leaves it untouched.
# Writes, per article, relations_ext_consolidated.json, and ONE global changes file listing ONLY the
# records that changed (before -> after). Pure bookkeeping. Run right after consolidate_vocab.
EXT_CONSOLIDATED_NAME = "relations_ext_consolidated.json"                       # per-article refreshed copy
EXT_CHANGES_PATH      = RESULTS_DIR / "relation_consolidation_ext_changes.json" # global "what changed" file
_CHANGED_FIELDS = ("vocab_id", "property", "property_definition", "property_parent")   # substantive fields

def refresh_ext_after_consolidation(report=None, verbose=True):
    if report is None:
        if not CONSOLIDATION_REPORT_PATH.exists():
            print("no consolidation report found — run consolidate_vocab first"); return
        report = json.loads(CONSOLIDATION_REPORT_PATH.read_text())
    remap = {m["id"]: r["survivor_id"] for r in report.get("merges", []) for m in r.get("merged", [])}
    by_id = {t["id"]: t for t in load_consolidated()["properties"]}   # survivors + re-parented properties

    all_changes, total = [], 0
    for fp in sorted(RESULTS_DIR.glob("*/relations_ext.json")):
        data = json.loads(fp.read_text())                            # the untouched incremental file
        aid  = data.get("article_id", fp.parent.name)
        art_changes = []
        for n in data["relations"]:
            v = n.get("vocab_id")
            if not v:
                continue
            s = by_id.get(remap.get(v, v))                           # follow a merge, else same id
            if s is None:
                continue
            old = {k: n.get(k) for k in _CHANGED_FIELDS}
            new = {"vocab_id": s["id"], "property": s["canonical_label"],
                   "property_definition": s.get("definition"), "property_parent": s.get("parent")}
            if new != old:
                n.update(new)                                        # mutate the in-memory COPY only
                art_changes.append({"subject_text": n.get("subject_text"), "object_text": n.get("object_text"),
                                    "before": old, "after": new})
        data["consolidated"], data["subprop_mode"] = True, SUBPROP_MODE
        (fp.parent / EXT_CONSOLIDATED_NAME).write_text(json.dumps(data, indent=2, ensure_ascii=False))
        if art_changes:
            all_changes.append({"article_id": aid, "n_changed": len(art_changes), "changes": art_changes})
            total += len(art_changes)
            if verbose:
                print(f"  {aid}: {len(art_changes)} record(s) changed")

    EXT_CHANGES_PATH.write_text(json.dumps({
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"), "subprop_mode": SUBPROP_MODE,
        "consolidated_vocab": str(CONSOLIDATED_VOCAB_PATH.name),
        "n_articles_changed": len(all_changes), "n_records_changed": total, "articles": all_changes,
    }, indent=2, ensure_ascii=False))
    print(f"done — wrote {EXT_CONSOLIDATED_NAME} for each article (incremental relations_ext.json untouched)\n"
          f"       {total} changed record(s) across {len(all_changes)} article(s) -> {EXT_CHANGES_PATH.name}")

refresh_ext_after_consolidation()

## Property hierarchy (schema.org + extension) — raw and consolidated, **separate** files

One indented tree of the property space — schema.org properties (linked by `subPropertyOf`) and the
extension (minted) properties together, each tagged `[schema.org]` or `[extension]`, sub-properties
nested under their parent. Built **twice**: `property_hierarchy_ext.txt` from the **raw incremental**
vocab, and `property_hierarchy_ext_consolidated.txt` from the **consolidated** vocab. Set
`INCLUDE_ALL_SCHEMA=True` to include every schema.org property, or leave it `False` to show just the
branches your extension properties hang off (extension properties + their super-property chains).

In [ ]:
# Property hierarchy: schema.org properties + extension (vocab) properties in ONE indented tree, each
# tagged [schema.org] or [extension]; sub-properties nested under their subPropertyOf parent.
#   property_hierarchy_ext.txt              <- raw incremental vocab (relation_vocabulary.json)
#   property_hierarchy_ext_consolidated.txt <- consolidated vocab   (re-parentings/merges reflected)
#   INCLUDE_ALL_SCHEMA=False -> only extension properties + their super-property chains.
#   INCLUDE_ALL_SCHEMA=True  -> the entire schema.org property set too (large).
INCLUDE_ALL_SCHEMA               = False
HIERARCHY_PATH                   = RESULTS_DIR / "property_hierarchy_ext.txt"                # raw incremental
HIERARCHY_CONSOLIDATED_PATH      = RESULTS_DIR / "property_hierarchy_ext_consolidated.txt"   # consolidated

def build_property_hierarchy(vocab, write_path, include_all_schema=INCLUDE_ALL_SCHEMA, show=50):
    schema_labels = {p["label"] for p in schema_pool}
    schema_parent = {p["label"]: _schema_super(p["label"]) for p in schema_pool}
    ext = {t["canonical_label"]: t for t in vocab["properties"]}

    parent_of, kind = {}, {}
    for lbl in schema_labels:
        parent_of[lbl] = schema_parent.get(lbl); kind[lbl] = "schema.org"
    for lbl, t in ext.items():                                   # extension overrides if same label
        parent_of[lbl] = t.get("parent"); kind[lbl] = "extension"

    if include_all_schema:
        included = set(schema_labels) | set(ext)
    else:                                                        # only extensions + their ancestor chains
        included = set(ext)
        for lbl in list(ext):
            cur, seen = parent_of.get(lbl), set()
            while cur and cur not in seen and cur not in included:
                seen.add(cur); included.add(cur); cur = parent_of.get(cur)

    children, roots = defaultdict(list), []
    for lbl in included:
        p = parent_of.get(lbl)
        (children[p].append(lbl) if p in included else roots.append(lbl))
    for p in children:
        children[p].sort()

    lines = []
    def walk(lbl, depth, seen):
        tag   = "[extension]" if kind.get(lbl) == "extension" else "[schema.org]"
        extra = f"  (support {ext[lbl].get('support', 0)})" if lbl in ext else ""
        lines.append("  " * depth + f"{lbl}  {tag}{extra}")
        for c in children.get(lbl, []):
            if c not in seen:
                walk(c, depth + 1, seen | {c})
    for r in sorted(set(roots)):
        walk(r, 0, {r})

    text = "\n".join(lines)
    if write_path:
        write_path.write_text(text)
    print(f"property hierarchy: {len(lines)} nodes ({len(ext)} extension) | "
          f"include_all_schema={include_all_schema} -> {write_path}\n" + "-" * 74)
    print(text if len(lines) <= show else "\n".join(lines[:show]) + f"\n  ... ({len(lines) - show} more — see file)")
    return text

# Raw incremental hierarchy (always available), then the consolidated one (reflects re-parentings/merges).
print("=== RAW incremental property vocabulary ===")
build_property_hierarchy(load_vocab(),        HIERARCHY_PATH)
print("\n=== CONSOLIDATED property vocabulary ===")
build_property_hierarchy(load_consolidated(), HIERARCHY_CONSOLIDATED_PATH)